<a href="https://colab.research.google.com/github/tamyriscabral/PFC2/blob/main/Anomalias.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# IMPORTS
# ============================================================
from pathlib import Path
import re
import unicodedata
import os
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [8]:
# ============================================================
# CONFIGURAÇÃO DE CAMINHOS E DIRETÓRIOS
# ============================================================
ARQUIVO_DADOS = Path("data.xlsx")
ARQUIVO_ALARMES = Path("data_alarmes.xlsm")  # Deixe None se não houver
ARQUIVO_LOCALIZACAO = Path("luminarias_com_localizacao_nova.csv")
PASTA_SAIDA = Path("/content/resultado_anomalias")
PASTA_SAIDA.mkdir(exist_ok=True, parents=True)

In [9]:
# ============================================================
# CONFIGURAÇÕES DE COLUNAS E LIMIARES
# ============================================================
COL = {
    "id": "ID Nox",
    "data": "Data e hora final",
    "consumo_real_wh": "Consumo Real (Wh)",
    "consumo_estimado_wh": "Consumo Estimado Calculado (kWh)",
    "potencia_cadastrada_w": "Potencia Cadastrada (W)",
    "tempo_ligado_min": "Tempo Ligado",
    "tempo_esperado_h": "Tempo Escuridão no Registro",
    "latitude": "Latitude",
    "longitude": "Longitude",
    "tipo_ponto": "tipo_ponto",
    "drc_planilha": "Desvio Relativo de Consumo",
    "drc_abs_planilha": "Desvio Relativo Absoluto",
    "drt_planilha": "Desvio relativo de tempo",
    "drp_planilha": "Desvio Relativo de Potência",
}

COL_ALARME = {
    "id": "Ponto",
    "tipo": "Tipo Alarme",
    "data": "Data",
}

LIMIAR_DRC = 10       # Divergência relativa de consumo (%)
LIMIAR_DRP = 10       # Divergência relativa de potência (%)
LIMIAR_DRT = 20       # Divergência relativa de tempo (%)

CONSUMO_CASO_KWH = 366
TOLERANCIA_CONSUMO_KWH = 2
POTENCIAS_CADASTRO = [30, 100]
PCC_FAIXA_30W = (70, 78)
PCC_ALVO_100W = 47

In [10]:
# ============================================================
# FUNÇÕES AUXILIARES
# ============================================================
def normalizar_nome(texto):
    texto = unicodedata.normalize("NFKD", str(texto))
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    return re.sub(r"[^a-z0-9]+", "", texto.lower())

def localizar_coluna(df, nome):
    if nome is None:
        return None
    mapa = {normalizar_nome(c): c for c in df.columns}
    return mapa.get(normalizar_nome(nome))

def exigir_coluna(df, chave):
    nome_configurado = COL[chave]
    encontrada = localizar_coluna(df, nome_configurado)
    if encontrada is None:
        raise KeyError(
            f"Não encontrei a coluna configurada para '{chave}': "
            f"'{nome_configurado}'.\nColunas disponíveis: {list(df.columns)}"
        )
    return encontrada

def numero(serie):
    if pd.api.types.is_numeric_dtype(serie):
        return pd.to_numeric(serie, errors="coerce")
    s = serie.astype("string").str.strip()
    s = s.str.replace(".", "", regex=False).str.replace(",", ".", regex=False)
    return pd.to_numeric(s, errors="coerce")

def salvar_csv(df, nome):
    df.to_csv(PASTA_SAIDA / nome, index=False, encoding="utf-8-sig")

def resumo_contagem(df, mascara, nome_classe, denominador_registros, denominador_luminarias):
    selecionado = df.loc[mascara]
    return {
        "Classe": nome_classe,
        "Registros": len(selecionado),
        "Luminarias": selecionado["_id"].nunique(),
        "% dos registros": (
            100 * len(selecionado) / denominador_registros
            if denominador_registros else np.nan
        ),
        "% das luminarias": (
            100 * selecionado["_id"].nunique() / denominador_luminarias
            if denominador_luminarias else np.nan
        ),
    }

def meses_por_luminaria(df, mascara, nome_classe):
    temp = df.loc[
        mascara & df["_mes"].notna(),
        ["_id", "_mes"]
    ].drop_duplicates()

    if temp.empty:
        return pd.DataFrame(columns=[
            "ID Nox", "Classe", "Meses com ocorrencia",
            "Meses registrados", "Razao ocorrencia/registro"
        ])

    ocorrencias = temp.groupby("_id")["_mes"].nunique().rename("Meses com ocorrencia")
    registrados = df.loc[df["_mes"].notna()].groupby("_id")["_mes"].nunique().rename("Meses registrados")

    out = pd.concat([ocorrencias, registrados], axis=1).dropna(subset=["Meses com ocorrencia"])
    out["Razao ocorrencia/registro"] = out["Meses com ocorrencia"] / out["Meses registrados"]
    out["Classe"] = nome_classe

    return out.reset_index().rename(columns={"_id": "ID Nox"})[[
        "ID Nox", "Classe", "Meses com ocorrencia",
        "Meses registrados", "Razao ocorrencia/registro"
    ]]

def normalizar_id_analise(serie):
    original = serie.astype("string").str.strip()
    numerico = pd.to_numeric(original, errors="coerce")
    resultado = original.copy()
    inteiros = numerico.notna() & np.isclose(numerico.fillna(0) % 1, 0)
    resultado.loc[inteiros] = numerico.loc[inteiros].astype("Int64").astype("string")
    return resultado

In [11]:
# ============================================================
# LEITURA E PROCESSAMENTO DOS DADOS PRINCIPAIS
# ============================================================
if not ARQUIVO_DADOS.exists():
    raise FileNotFoundError(
        f"Arquivo não encontrado: {ARQUIVO_DADOS.resolve()}. "
        "Ajuste o caminho na seção de configuração."
    )

df = pd.read_excel(ARQUIVO_DADOS, sheet_name="Planilha1")
df.columns = [str(c).strip() for c in df.columns]

df_localizacao = pd.read_csv(ARQUIVO_LOCALIZACAO)
df_localizacao = df_localizacao[["ID Nox", "tipo_ponto"]].copy()

df["ID Nox"] = df["ID Nox"].astype("string").str.strip()
df_localizacao["ID Nox"] = df_localizacao["ID Nox"].astype("string").str.strip()

df = df.merge(df_localizacao, on="ID Nox", how="left", validate="many_to_one")

c_id = exigir_coluna(df, "id")
c_data = exigir_coluna(df, "data")
c_cr = exigir_coluna(df, "consumo_real_wh")
c_cec = exigir_coluna(df, "consumo_estimado_wh")
c_pc = exigir_coluna(df, "potencia_cadastrada_w")
c_tl = exigir_coluna(df, "tempo_ligado_min")
c_te = exigir_coluna(df, "tempo_esperado_h")

df["_id"] = df[c_id].astype("string").str.strip()
df["_data"] = pd.to_datetime(df["Data e hora final"], format="mixed", errors="coerce")
df["_mes"] = df["_data"].dt.to_period("M").astype("string")

df["_CR_Wh"] = numero(df[c_cr])
df["_CEC_Wh"] = numero(df[c_cec]) * 1000
df["_PC_W"] = numero(df[c_pc])
df["_TL_min"] = numero(df[c_tl])
df["_TL_h"] = df["_TL_min"] / 60
df["_TE_h"] = numero(df[c_te])

df["_DRC"] = np.where(df["_CEC_Wh"] > 0, (df["_CR_Wh"] - df["_CEC_Wh"]) / df["_CEC_Wh"] * 100, np.nan)
df["_PCC_W"] = np.where(df["_TL_h"] > 0, df["_CR_Wh"] / df["_TL_h"], np.nan)
df["_DRP"] = np.where(df["_PC_W"] > 0, (df["_PCC_W"] - df["_PC_W"]) / df["_PC_W"] * 100, np.nan)
df["_DRT"] = np.where(df["_TE_h"] > 0, (df["_TL_h"] - df["_TE_h"]) / df["_TE_h"] * 100, np.nan)

df = df[df["_id"].notna() & df["_id"].ne("")].copy()
n_registros = len(df)
n_luminarias = df["_id"].nunique()

df_excluir = df.copy()
df = df[df["Manter/excluir"].astype("string").str.strip().str.casefold().eq("manter")].copy()

ids_luminarias_1w = set(df.loc[df["_PC_W"].eq(1), "_id"].dropna().unique())
mascara_sem_1w = ~df["_id"].isin(ids_luminarias_1w)
df["_DRC_sem_1W"] = df["_DRC"].where(mascara_sem_1w)
df["_DRP_sem_1W"] = df["_DRP"].where(mascara_sem_1w)

mascara_tempo_invalido = (df["Tempo Ligado"].isna() | (df["Tempo Ligado"] == 0)) & df["_CR_Wh"].fillna(0).ne(0)
df_tempo_invalido = df.loc[mascara_tempo_invalido].copy()
df_tempo_invalido.to_csv(PASTA_SAIDA / "registros_tempo_ausente_ou_zero.csv", index=False, encoding="utf-8-sig")

In [12]:
# ============================================================
# CLASSES DE ANÁLISE
# ============================================================
mascara_1w = df["_PC_W"].eq(1)
c_lat = localizar_coluna(df, COL["latitude"])
c_lon = localizar_coluna(df, COL["longitude"])

if c_lat and c_lon:
    lat = numero(df[c_lat])
    lon = numero(df[c_lon])
    mascara_sem_coord = lat.isna() | lon.isna() | (lat.eq(0) & lon.eq(0))
else:
    mascara_sem_coord = pd.Series(False, index=df.index)

mascara_drc_pos = df["_DRC"] > LIMIAR_DRC
mascara_drc_neg = df["_DRC"] < -LIMIAR_DRC
mascara_drp_pos = df["_DRP"] > LIMIAR_DRP
mascara_drp_neg = df["_DRP"] < -LIMIAR_DRP
mascara_drt_pos = df["_DRT"] > LIMIAR_DRT
mascara_drt_neg = df["_DRT"] < -LIMIAR_DRT

classes = [
    ("cadastro_potencia_1w", mascara_1w),
    ("cadastro_tempo_invalido", mascara_tempo_invalido),
    ("cadastro_coordenada_ausente", mascara_sem_coord),
    ("consumo_drc_positivo", mascara_drc_pos),
    ("consumo_drc_negativo", mascara_drc_neg),
    ("potencia_drp_positivo", mascara_drp_pos),
    ("potencia_drp_negativo", mascara_drp_neg),
    ("tempo_drt_positivo", mascara_drt_pos),
    ("tempo_drt_negativo", mascara_drt_neg),
]

linhas = [resumo_contagem(df, mascara, nome, n_registros, n_luminarias) for nome, mascara in classes]
tabela_classes = pd.DataFrame(linhas)
salvar_csv(tabela_classes, "01_resumo_classes.csv")

for nome, mascara in classes:
    df.loc[mascara].to_csv(PASTA_SAIDA / f"registros_{nome}.csv", index=False, encoding="utf-8-sig")

In [13]:
# ============================================================
# ANÁLISE DE ALARMES
# ============================================================
alarmes = None
alarmes_por_tipo = []
relacoes = []

if ARQUIVO_ALARMES is not None and Path(ARQUIVO_ALARMES).exists():
    alarmes = pd.read_excel(ARQUIVO_ALARMES, sheet_name="Export")
    alarmes.columns = [str(c).strip() for c in alarmes.columns]

    a_id = localizar_coluna(alarmes, ["Ponto"])
    a_tipo = localizar_coluna(alarmes, ["Tipo alarme"])
    a_inicio = localizar_coluna(alarmes, ["Data do primeiro alarme"])
    a_fim = localizar_coluna(alarmes, ["Data do ultimo alarme"])
    a_dias = localizar_coluna(alarmes, ["Dias De Medicao"])
    a_qtd = localizar_coluna(alarmes, ["Quantidade de alarmes no periodo"])

    alarmes["_id"] = pd.to_numeric(alarmes[a_id], errors="coerce").astype("Int64").astype("string")
    alarmes["_tipo"] = alarmes[a_tipo].astype("string").str.strip().str.lower()
    alarmes["_data_inicio"] = pd.to_datetime(alarmes[a_inicio], errors="coerce", dayfirst=False)
    alarmes["_data_fim"] = pd.to_datetime(alarmes[a_fim], errors="coerce", dayfirst=False)
    alarmes["_dias_medicao"] = pd.to_numeric(alarmes[a_dias], errors="coerce")
    alarmes["_qtd_alarmes"] = pd.to_numeric(alarmes[a_qtd], errors="coerce")
    alarmes["_media_dia"] = alarmes["_qtd_alarmes"] / alarmes["_dias_medicao"].replace(0, np.nan)

    termos = {
        "offline": ["offline"],
        "apagada à noite": ["apagado à noite"],
        "acesa de dia": ["aceso durante o dia"],
    }

    for classe, variantes in termos.items():
        m = alarmes["_tipo"].isin([v.lower() for v in variantes])
        sub = alarmes.loc[m]
        alarmes_por_tipo.append({
            "Tipo de alarme": classe,
            "Luminárias": sub["_id"].nunique(),
            "Registros na planilha": len(sub),
            "Total de alarmes no período": sub["_qtd_alarmes"].sum(),
            "Dias de medição": sub["_dias_medicao"].sum(),
            "Média de alarmes por dia": sub["_media_dia"].mean(),
        })

    resumo_alarmes = pd.DataFrame(alarmes_por_tipo)
    salvar_csv(resumo_alarmes, "02_resumo_alarmes.csv")


In [14]:
# ============================================================
# SOBREPOSIÇÃO E PERSISTÊNCIA
# ============================================================
flags = pd.DataFrame(index=sorted(df["_id"].dropna().unique()))
flags.index.name = "ID Nox"

def flag_por_luminaria(mascara, nome):
    ids = df.loc[mascara, "_id"].dropna().unique()
    flags[nome] = flags.index.isin(ids)
    return ids

flag_por_luminaria(mascara_1w, "Cadastro 1 W")
flag_por_luminaria(mascara_tempo_invalido, "Cadastro TL inválido")
flag_por_luminaria(mascara_sem_coord, "Cadastro sem coordenada")
flag_por_luminaria(mascara_drc_pos | mascara_drc_neg, "Consumo")
flag_por_luminaria(mascara_drp_pos | mascara_drp_neg, "Potencia")
flag_por_luminaria(mascara_drt_pos | mascara_drt_neg, "Tempo")

flags["Numero de classes"] = flags[list(flags.columns)].sum(axis=1)
flags["Duas ou mais classes"] = flags["Numero de classes"] >= 2
salvar_csv(flags.reset_index(), "04_sobreposicao_por_luminaria.csv")

print("Execução de anomalias finalizada com sucesso.")

Execução de anomalias finalizada com sucesso.
